# Low-frequency noise analysis

Simple Speckit ASD workflow for rp_ads1278 CSV logs. Change `MEASUREMENT_PATH` and `CHANNEL` for other recordings.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from speckit import compute_spectrum, compute_single_bin

In [3]:
from pathlib import Path

NOT_MOD_PATH = Path("../logged_files/Better-wiring/NoNTC/NoNTC-Gain4-NoMOD.csv")
DEMOD_PATH = Path("../logged_files/Better-wiring/NoNTC/MOD_Gain4_NoNTC_v1.csv")
REFERENCE_VOLTS = 2.5

RP_ADC_CLOCK_HZ = 125_000_000.0
ADS1278_CLOCKS_PER_SAMPLE = 512.0
CODE_SCALE = 1 << 23

In [4]:
def load(path, channel, decimate=False):
    df = pd.read_csv(path)
    extclk_div = int(df["extclk_div"].iloc[0])
    mod_div = int(df["mod_div"].iloc[0])
    fs_adc = RP_ADC_CLOCK_HZ / (2.0 * extclk_div) / ADS1278_CLOCKS_PER_SAMPLE
    f_mod = RP_ADC_CLOCK_HZ / (2.0 * mod_div)

    codes = df[channel].to_numpy(dtype=float)
    if decimate:
        # CH8 updates ~10 Hz and is held — keep one sample per change
        idx = np.concatenate([[0], np.flatnonzero(np.diff(codes) != 0) + 1])
        codes = codes[idx]
        fs = f_mod
    else:
        fs = fs_adc

    volts = codes * (REFERENCE_VOLTS / CODE_SCALE)
    volts = volts - volts.mean()
    print(f"{path.name} {channel}: n={len(volts):,}, fs={fs:.2f} Hz")
    return volts, fs

speckit_kw = dict(Jdes=900, Kdes=100, order=0, win="Kaiser", psll=200, verbose=False)

volts_raw, fs_raw = load(NOT_MOD_PATH, "ch1")
volts_demod, fs_demod = load(DEMOD_PATH, "ch8", decimate=True)

asd_raw = compute_spectrum(volts_raw, fs=fs_raw, **speckit_kw)
asd_demod = compute_spectrum(volts_demod, fs=fs_demod, **speckit_kw)

NoNTC-Gain4-NoMOD.csv ch1: n=7,985,907, fs=8138.02 Hz


FileNotFoundError: [Errno 2] No such file or directory: '../logged_files/Better-wiring/NoNTC/MOD_Gain4_NoNTC_v1.csv'

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(asd_raw.f, asd_raw.asd, color="C3", label="NotModulated ch1")
ax.loglog(asd_demod.f, asd_demod.asd, color="k", label="Demodulated ch8 (decimated)")
ax.set_xlim(1e-4, 5)   # demod Nyquist ≈ 5 Hz at 10 Hz MOD
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel(r"ASD (V/$\sqrt{\rm Hz}$)")
ax.legend()
ax.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = asd_demod.plot(which="asd", errors=True, sigma=3)
ax.set_ylim(1e-8, 1e-4)
plt.show()